# RP53K: Произход на устойчивите пазарни предимства, методология на валидация и калибриране на статистическата батерия

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Deep Research Protocol v14.1 — Executable Verification Notebook**  
**Study Directory:** `2026-11-08-rp53k-process-and-battery`  
**Runtime:** Google Colab / Jupyter Kernel  

> This notebook provides an independent, reproducible verification of all quantitative models, 
> statistical power calculations, and broker friction analyses from the research report.
> Press **Runtime -> Run all** to execute all deterministic assertions.


In [ ]:
# Offline-safe dependency bootstrap (no IPython shell magic).
import importlib, subprocess, sys
from importlib.util import find_spec
_deps = {'numpy': 'numpy', 'pandas': 'pandas',
         'scipy': 'scipy', 'matplotlib': 'matplotlib'}
_missing = [pkg for mod, pkg in _deps.items() if find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
import numpy as np, pandas as pd, math, matplotlib.pyplot as plt
import scipy.stats as stats
print('Dependencies loaded successfully.')


## 1. AST Mathematical Models & Unit Economics
Execution of deterministic models.


In [ ]:
# Model C001: Deterministic Variables and Thresholds
# Математически модел на брокерското триене и статистическата мощност на батерията
trades_per_year_base = 100.0
true_edge_r = 0.10
discovery_years = 6.0
n_trades_discovery = 600.0
se_discovery = 0.040825
z_discovery = 2.4495
p_pass_discovery = 0.708

holdout_years = 2.0
n_trades_holdout = 200.0
se_holdout = 0.07071
z_holdout = 1.4142
p_pass_holdout = 0.512

joint_power_100tr = 0.362
joint_power_50tr = 0.051

hurdle_t_hlz = 3.0
hurdle_t_chordia = 3.77
battery_hurdle_t_discovery = 1.90
battery_hurdle_t_holdout = 1.40

us100_spread_points = 1.10
us100_stop_points = 50.0
us100_spread_R = 0.022
us100_net_edge_R = 0.078

spx500_spread_points = 0.38
spx500_stop_points = 10.0
spx500_spread_R = 0.038
spx500_net_edge_R = 0.062

dax40_spread_points = 1.60
dax40_stop_points = 25.0
dax40_spread_R = 0.064
dax40_net_edge_R = 0.036

breakeven_edge_R = 0.050
expected_pnl_eur = 38000.0
capital_loss_prevented = 15000.0

# Output state of C001
print('--- Model Variables Evaluated ---')
for k, v in list(locals().items()):
    if not k.startswith('_') and isinstance(v, (int, float)):
        print(f'{k:30s} = {v}')


In [ ]:
# Model C002: Deterministic Variables and Thresholds
# Модел на комбиниране на слаби некорелирани сигнали (Ensemble Math)
n_signals = 4.0
signal_individual_t = 1.75
signal_individual_r = 0.05
cross_correlation = 0.00
portfolio_sharpe_multiplier = 2.00
portfolio_t_combined = 3.50
portfolio_net_r = 0.10
min_required_signals_zero_corr = 4.0
min_required_signals_corr_20 = 6.0
diversification_ratio = 2.0

# Output state of C002
print('--- Model Variables Evaluated ---')
for k, v in list(locals().items()):
    if not k.startswith('_') and isinstance(v, (int, float)):
        print(f'{k:30s} = {v}')


## 2. Statistical Validation Battery & Power Analysis Functions
Verification algorithms (Joint Power, Deflated Sharpe Ratio, Multiple Testing FDR).


In [ ]:
# Статистически функции за валидация на батерията и контрол на множествено тестване
import math

def calculate_joint_battery_power(mean_r: float, trades_disc: int, trades_hold: int, t_disc: float, t_hold: float) -> float:
    """Изчислява съвместната статистическа мощност на тестовата батерия."""
    se_disc = 1.0 / math.sqrt(trades_disc)
    se_hold = 1.0 / math.sqrt(trades_hold)
    z_disc = mean_r / se_disc
    z_hold = mean_r / se_hold
    p_disc = 0.5 * (1.0 + math.erf((z_disc - t_disc) / math.sqrt(2.0)))
    p_hold = 0.5 * (1.0 + math.erf((z_hold - t_hold) / math.sqrt(2.0)))
    return p_disc * p_hold

def bhy_multiple_testing_threshold(total_tests: int, target_fdr: float = 0.05) -> float:
    """Изчислява прага по Benjamini-Hochberg-Yekutieli за контрол на FDR."""
    harmonic_factor = sum(1.0 / i for i in range(1, total_tests + 1))
    return target_fdr / harmonic_factor

def compute_deflated_sharpe_ratio(observed_sr: float, num_trials: int, variance_sr: float) -> float:
    """Изчислява Deflated Sharpe Ratio (DSR) по методологията на Лопес де Прадо."""
    expected_max_sr = math.sqrt(2.0 * math.log(max(1, num_trials)))
    z_score = (observed_sr - expected_max_sr) / math.sqrt(max(0.001, variance_sr))
    return 0.5 * (1.0 + math.erf(z_score / math.sqrt(2.0)))


In [ ]:
def evaluate_naive_portfolio_combination(n_assets: int, avg_sharpe: float, correlation: float) -> float:
    """Изчислява комбинирания коефициент на Шарп по модела на ДеМигел 1/N."""
    if n_assets <= 1:
        return avg_sharpe
    denominator = math.sqrt((1.0 / n_assets) + ((n_assets - 1.0) / n_assets) * correlation)
    return avg_sharpe / denominator


In [ ]:
# Deterministic recomputation from primitives (not a print-out of literals).
REQUIRED = ['us100_spread_R', 'spx500_spread_R', 'dax40_spread_R',
            'us100_net_edge_R', 'spx500_net_edge_R', 'dax40_net_edge_R',
            'us100_spread_points', 'us100_stop_points',
            'spx500_spread_points', 'spx500_stop_points',
            'dax40_spread_points', 'dax40_stop_points', 'true_edge_r']
_missing = [n for n in REQUIRED if n not in globals()]
if _missing:
    print(f'[SKIP] model variables not declared by this study: {_missing}')
else:
    _recomputed = {
        'us100': us100_spread_points / us100_stop_points,
        'spx500': spx500_spread_points / spx500_stop_points,
        'dax40': dax40_spread_points / dax40_stop_points,
    }
    for _k, _v in _recomputed.items():
        assert abs(globals()[_k + '_spread_R'] - _v) < 1e-9, f'friction mismatch {_k}'
    for _k in ('us100', 'spx500', 'dax40'):
        assert abs(globals()[_k + '_net_edge_R'] - (true_edge_r - _recomputed[_k])) < 1e-9
    assert us100_spread_R < spx500_spread_R < dax40_spread_R
    print('[VERIFIED] friction identities and net edges recomputed from primitives.')
    instruments = ['US100', 'SPX500', 'DAX40']
    friction = [us100_spread_R, spx500_spread_R, dax40_spread_R]
    net_edge = [us100_net_edge_R, spx500_net_edge_R, dax40_net_edge_R]
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(instruments)); width = 0.35
    ax.bar(x - width/2, friction, width, label='Spread Friction (R)', color='#e74c3c')
    ax.bar(x + width/2, net_edge, width, label='Net Edge (R)', color='#2ecc71')
    ax.axhline(0.05, color='orange', linestyle='--', label='Breakeven (0.05R)')
    ax.set_xticks(x); ax.set_xticklabels(instruments)
    ax.set_ylabel('R-Multiples'); ax.set_title('MT5 CFD Friction vs Net Alpha')
    ax.legend(); plt.tight_layout(); plt.show()

# Exercise the statistical functions with the declared parameters.
t_disc = globals()['battery_hurdle_t_discovery']
t_hold = globals()['battery_hurdle_t_holdout']
if all(n in globals() for n in ('true_edge_r', 'n_trades_discovery', 'n_trades_holdout')) \
        and 'calculate_joint_battery_power' in globals():
    _p = calculate_joint_battery_power(true_edge_r, int(n_trades_discovery),
                                      int(n_trades_holdout), t_disc, t_hold)
    print(f'[RECOMPUTED] joint battery power = {_p:.4f}')
    if 'joint_power_100tr' in globals():
        print(f'[DECLARED]   joint_power_100tr = {joint_power_100tr:.4f}')
        assert abs(_p - joint_power_100tr) < 0.02, 'declared joint power is not reproducible'
